In [ ]:
%%capture
!pip install torchvision
!pip install torchmetrics
!pip install pytorchcocotools
!pip install torch

In [ ]:
import os
import glob
import torch
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import subprocess

import torchvision
import torchvision.transforms.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms import ToTensor
from pytorchcocotools.coco import COCO
from torchmetrics.classification import BinaryF1Score
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# ===============================
# DATASET
# ===============================
class DropletCocoDataset(Dataset):
    def __init__(self, img_dir, json_path):
        self.coco = COCO(json_path)
        self.img_dir = img_dir

        self.ids = [
            img_id for img_id in self.coco.imgs
            if os.path.exists(os.path.join(img_dir, self.coco.loadImgs(img_id)[0]["file_name"]))
            and len(self.coco.getAnnIds(imgIds=img_id)) > 0
        ]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]

        img_path = os.path.join(self.img_dir, img_info["file_name"])
        image = Image.open(img_path).convert("RGB")

        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))

        boxes, masks, areas = [], [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            boxes.append([x, y, x + w, y + h])
            masks.append(self.coco.annToMask(ann))
            areas.append(ann["area"])

        boxes = torch.tensor(boxes, dtype=torch.float32)
        masks = torch.as_tensor(np.stack(masks), dtype=torch.uint8)
        areas = torch.tensor(areas, dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": torch.ones((len(boxes),), dtype=torch.int64),
            "masks": masks,
            "area": areas,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
            "image_id": torch.tensor([img_id])
        }

        image = F.to_tensor(image)
        return image, target

# ===============================
# MODEL
# ===============================
def load_model(model_path, device, num_classes=2):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn_v2(weights=None)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    return model

# ===============================
# HELPERS
# ===============================
def collate_fn(batch):
    return tuple(zip(*batch))


def merge_instance_masks(output, score_thresh=0.6, mask_thresh=0.2):
    masks = output['masks']
    scores = output['scores']

    keep = scores > score_thresh
    masks = masks[keep]

    if len(masks) == 0:
        return torch.zeros((output['masks'].shape[2], output['masks'].shape[3]), device=masks.device)

    masks = masks.squeeze(1)
    merged = masks.max(dim=0)[0]
    return (merged > mask_thresh).float()

# ===============================
# TRAINING
# ===============================
def train_model(model, train_loader, device, epochs=5):
    model.train()

    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=0.001, momentum=0.9, weight_decay=0.0005
    )

    for epoch in range(epochs):
        total_loss = 0

        for images, targets in train_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

# ===============================
# EVALUATION
# ===============================
def evaluate_model(model, test_loader, device):
    f1_metric = BinaryF1Score(threshold=0.5).to(device)
    map_metric = MeanAveragePrecision(iou_type="segm").to(device)

    total_fp, total_tn = 0, 0

    with torch.no_grad():
        for images, targets in test_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            outputs_for_map, targets_for_map = [], []

            for output, target in zip(outputs, targets):
                pred_mask = merge_instance_masks(output).to(device)
                gt_mask = target['masks'].any(0).float().to(device)

                pred_binary = pred_mask > 0.5
                gt_binary = gt_mask > 0.5

                total_fp += ((pred_binary == 1) & (gt_binary == 0)).sum().item()
                total_tn += ((pred_binary == 0) & (gt_binary == 0)).sum().item()

                f1_metric.update(pred_mask.flatten(), gt_mask.flatten())

                outputs_for_map.append({
                    'masks': (output['masks'] > 0.5).to(torch.uint8),
                    'boxes': output['boxes'],
                    'scores': output['scores'],
                    'labels': torch.ones(len(output['boxes']), dtype=torch.int64, device=device)
                })

                targets_for_map.append({
                    'masks': target['masks'].to(torch.uint8),
                    'boxes': target['boxes'],
                    'labels': torch.ones(len(target['boxes']), dtype=torch.int64, device=device)
                })

            map_metric.update(outputs_for_map, targets_for_map)

    return {
        "F1": f1_metric.compute().item(),
        "mAP": map_metric.compute()['map'].item(),
        "mAP50": map_metric.compute()['map_50'].item(),
        "FPR": total_fp / (total_fp + total_tn + 1e-8)
    }

# ===============================
# AST STAGE 2 (FULL PIPELINE)
# ===============================
def infer_image(model, img_path, device, score_thresh=0.5, mask_thresh=0.5):
    img = Image.open(img_path).convert("RGB")
    tensor = F.to_tensor(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(tensor)[0]

    keep = output['scores'] > score_thresh
    boxes = output['boxes'][keep].cpu().numpy()
    masks = (output['masks'][keep, 0] > mask_thresh).cpu().numpy()

    return np.array(img), boxes, masks


def crop_droplet(img, box, pad=10):
    H, W = img.shape[:2]
    x1, y1, x2, y2 = box.astype(int)

    x1, y1 = max(0, x1-pad), max(0, y1-pad)
    x2, y2 = min(W, x2+pad), min(H, y2+pad)

    return img[y1:y2, x1:x2]


def build_droplet_dirs(image_paths, model, device, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    ref_boxes = None

    for t, img_path in enumerate(sorted(image_paths)):
        img, boxes, masks = infer_image(model, img_path, device)

        if ref_boxes is None:
            ref_boxes = boxes
            for i in range(len(boxes)):
                (output_dir / f"droplet_{i:03d}").mkdir(exist_ok=True)

        for i, box in enumerate(boxes):
            crop = crop_droplet(img, box)
            save_path = output_dir / f"droplet_{i:03d}" / f"t{t:03d}.png"
            Image.fromarray(crop).save(save_path)

    return {i: output_dir / f"droplet_{i:03d}" for i in range(len(ref_boxes))}


def run_colonyscanalyser(droplet_dirs):
    results = {}

    for did, path in droplet_dirs.items():
        subprocess.run(["scanalyser", str(path)])
        csvs = list(Path(path).rglob("colony_data.csv"))
        if csvs:
            results[did] = csvs[0]

    return results


def classify_droplets(results_paths, threshold=500):
    out = {}

    for did, csv_path in results_paths.items():
        try:
            df = pd.read_csv(csv_path)
            col = next((c for c in df.columns if "area" in c.lower()), df.columns[-1])
            out[did] = "RESISTANT" if df[col].max() > threshold else "SUSCEPTIBLE"
        except:
            out[did] = "SUSCEPTIBLE"

    return out

def get_transform():
    return ToTensorDet()

import torchvision.transforms.functional as F

class ToTensorDet:
    def __call__(self, image, target):
        image = F.to_tensor(image)
        return image, target


def population_call(classifications, thresh=0.1):
    vals = list(classifications.values())
    r = vals.count("RESISTANT")
    frac = r / len(vals) if vals else 0
    return "RESISTANT" if frac >= thresh else "SUSCEPTIBLE"


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---- DATA ----
    cipro_genta_dataset = DropletCocoDataset(
        img_dir="/content/drive/MyDrive/Droplet Segmentation/Freeman_Training/Cipro_Genta",
        json_path="/content/drive/MyDrive/Droplet Segmentation/Freeman_Training/Cipro_Genta/cipro_genta_ann.json",
        transforms=ToTensorDet()
    )

    pip_dataset = DropletCocoDataset(
        img_dir="/content/drive/MyDrive/Droplet Segmentation/Freeman_Training/Pip",
        json_path="/content/drive/MyDrive/Droplet Segmentation/Freeman_Training/Pip/instances_default.json",
        transforms=ToTensorDet()
    )

    train_dataset = ConcatDataset([cipro_genta_dataset, pip_dataset])

    test_dataset = DropletCocoDataset(
      img_dir="/content/drive/MyDrive/Droplet Segmentation/Freeman_Validation",
      json_path="/content/drive/MyDrive/Droplet Segmentation/Freeman_Validation/instances_default.json",
      transforms=get_transform())

    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

    # ---- MODEL ----
    model = load_model("model.pth", device)

    # ---- TRAIN ----
    train_model(model, train_loader, device)

    torch.save(model.state_dict(), "model.pth")

    # ---- EVAL ----
    metrics = evaluate_model(model, test_loader, device)
    print(metrics)

    # ---- AST ----
    images = glob.glob("/path/timepoints/*.jpg")
    droplet_dirs = build_droplet_dirs(images, model, device, "./droplets")

    results = run_colonyscanalyser(droplet_dirs)
    classifications = classify_droplets(results)

    final_call = population_call(classifications)

    print("\nAST RESULT:", final_call)


TypeError: DropletCocoDataset.__init__() got an unexpected keyword argument 'transforms'